# UMUD R71T — lepszy model pęczków + symulator protokołu (Colab, GPU)

**Jak uruchomić:** GPU (najlepiej **L4**; T4 też działa, ok. 2× wolniej) → _Środowisko wykonawcze → Uruchom wszystko_.
Po rozłączeniu znów _Uruchom wszystko_ — to, co policzone, zostanie pominięte, trening wznowi się od ostatniego checkpointu.
**Zmieniaj tylko komórkę 1.** Komórka 2 zawiera spakowany kod sprawdzany sumą SHA‑256 (edycja go uszkadza).

**Po co.** Na LB granicą jest szum pojedynczych fragmentów pęczków (3 pęczki zamiast 5: +0,016). Ten notebook:
1. **Etap S — symulator protokołu (ok. 40 min).** Na 2046 obrazach treningowych z narysowanymi pęczkami liczy fragmenty R22
   *poza foldem* (każdy obraz modelem, który go nie widział), rozcięgna R12B i v2 oraz linie pęczków z masek ekspertów.
   Z tego offline policzę „etykietę” jak w protokole (3 najmniej ekstrapolowane narysowane pęczki) i sprawdzę każdą regułę
   na 2000 obrazach zamiast na LB. Do tego eksport fragmentów testu i OSF z pełną geometrią, kątem z pola kierunku R22
   (którego dotąd nie używaliśmy dla fragmentów) i pewnością modelu.
2. **Etap T — trening nowego modelu pęczków F71 (ok. 2 h na fold na L4).** Ta sama architektura co R22, ale wejście 1024 px
   zamiast 768, cel = rurka wokół narysowanych linii + człon ciągłości (odzysk szkieletu) → dłuższe fragmenty, kierunek każdej
   linii. Te same foldy co R22, więc porównanie na tych samych obrazach. Po każdym foldzie fragmenty poza foldem.
3. **Etap E — fragmenty testu i OSF z F71** po 1, 3 i 5 ukończonych foldach (ok. 10 min każdy).

| etap | czas (L4) | wynik na Drive (`UMUD_Challenge/outputs/R71T_v1_0_0/`) |
|---|---|---|
| S | ok. 40 min | `S/train_*.csv.gz`, `S/test_*.csv.gz`, `S/osf_*.csv.gz` (fragmenty osobno dla progów 0,4 / 0,5 / 0,6), `S/S_SUMMARY.json` |
| T | ok. 2 h / fold (5 foldów) | `T/fold_k/` (model, historia, fragmenty poza foldem) |
| E | ok. 10 min po 1., 3., 5. foldzie | `E1/`, `E3/`, `E5/` |

Cache danych treningowych budowany jest raz (podczas etapu S) i zapisywany na Drive jako jeden plik — kolejne sesje
rozpakowują go w kilka minut. **Co mi przysłać:** informację, który etap się skończył (wystarczy ostatnia linia wydruku) —
pliki odczytam z Drive. Najważniejszy jest etap S: z nim od razu zrobię nowe sondy LB. Nie uruchamiaj dwóch sesji na tym
samym `RUN_NAME` jednocześnie.


In [ ]:
# =====================================================================================
# KOMÓRKA 1 — USTAWIENIA (tylko tę komórkę można zmieniać)
# =====================================================================================
RUN_NAME = 'R71T_v1_0_0'     # ten sam katalog = wznowienie; nowy = nowy eksperyment
RUN_S = True                 # etap S (symulator + eksport R22); pomijany, jeśli już policzony
RUN_TRAIN = True             # etap T (+ eksporty E po EXPORT_AFTER ukończonych foldach)
FOLDS = [2, 3, 4, 0, 1]      # kolejność foldów (2 pierwszy: ma najwięcej grup)
EXPORT_AFTER = [1, 3, 5]     # eksport E po tylu ukończonych foldach
SESSION_HOURS = 11.0         # po tylu godzinach sesji nie zaczyna nowej epoki (checkpoint zostaje)
CHECK_IMAGE_SHA = True       # kontrola sum obrazów testu i OSF


In [ ]:
# =====================================================================================
# KOMÓRKA 2 — KOD. NIE EDYTUJ: spakowany kod sprawdzany sumą SHA-256 (ustawienia są w komórce 1).
# =====================================================================================
import os, sys, json, time, base64, zlib, hashlib, importlib.util, subprocess, shutil
from pathlib import Path
T_START = time.time()
FIXTURE = os.environ.get('R71T_FIXTURE_ROOT')          # wyłącznie testy lokalne
if FIXTURE:
    DRIVE_ROOT = Path(FIXTURE) / 'MyDrive'; LOCAL = Path(FIXTURE) / 'content'
else:
    from google.colab import drive
    drive.mount('/content/drive')
    DRIVE_ROOT = Path('/content/drive/MyDrive'); LOCAL = Path('/content')
PROJECT = DRIVE_ROOT / 'UMUD_Challenge'
RUN = PROJECT / 'outputs' / RUN_NAME; RUN.mkdir(parents=True, exist_ok=True)
SRC = LOCAL / 'r71t_src'
PACKAGE_SHA = 'c8d3a28185abd3d303660b97bac327a498e637d78669c9b7d73caf656392fb24'
PAYLOAD = '...'

def _unpack(b64):
    raw = zlib.decompress(base64.b64decode(b64))
    if hashlib.sha256(raw).hexdigest() != PACKAGE_SHA:
        raise ValueError('SHA-256 kodu się nie zgadza')
    return raw

_backup = RUN / 'code' / f'r71t_{PACKAGE_SHA[:16]}.b64'
try:
    raw = _unpack(PAYLOAD)
    if not _backup.exists():
        _backup.parent.mkdir(parents=True, exist_ok=True); _backup.write_text(PAYLOAD)
except Exception as _e:
    if _backup.exists():
        raw = _unpack(_backup.read_text())
        print('UWAGA: kod w tej komórce był uszkodzony — użyłem nienaruszonej kopii z Drive:', _backup)
    else:
        raise RuntimeError('Kod w tej komórce jest uszkodzony (zwykle po edycji). Wczytaj świeży plik notebooka '
                           'i zmieniaj tylko komórkę 1.') from _e
for rel, content in json.loads(raw).items():
    p = SRC / rel; p.parent.mkdir(parents=True, exist_ok=True); p.write_text(content)
for module, pip_name in [('cv2', 'opencv-python-headless'), ('skimage', 'scikit-image'), ('timm', 'timm')]:
    if importlib.util.find_spec(module) is None:
        subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', pip_name])
sys.path.insert(0, str(SRC))
import numpy as np, pandas as pd, torch
import importlib as _il
import r45_core, r45_models, r71_geom, r71_sim, r71_train
for _m in (r45_core, r45_models, r71_geom, r71_sim, r71_train):   # po ponownym uruchomieniu: zawsze świeży kod
    _il.reload(_m)
DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
if DEVICE == 'cpu' and not FIXTURE:
    raise RuntimeError('Brak GPU. Środowisko wykonawcze → Zmień typ środowiska wykonawczego → GPU, potem Uruchom wszystko.')
GPU = torch.cuda.get_device_name(0) if DEVICE == 'cuda' else 'cpu'
if DEVICE == 'cuda':
    torch.backends.cudnn.benchmark = True

def log(*a):
    msg = ' '.join(str(x) for x in a); print(msg, flush=True)
    with open(RUN / 'LOG.txt', 'a') as fh:
        fh.write(time.strftime('%Y-%m-%d %H:%M:%S ') + msg + '\n')

def show(df, n=40):
    print(df.head(n).to_string(index=False), flush=True)

def cuda_gc():
    import gc
    gc.collect()
    if DEVICE == 'cuda':
        torch.cuda.empty_cache()

log('R71T start | GPU:', GPU, '| trening w', str(r71_train.precision_mode(DEVICE)[0]).replace('torch.', ''),
    '| torch', torch.__version__, '| kod', PACKAGE_SHA[:12], '| RUN', RUN)


In [ ]:
# =====================================================================================
# KOMÓRKA 3 — DANE I CHECKPOINTY (kontrola sum)
# =====================================================================================
ASSETS = SRC / 'assets'
R22RUN = PROJECT / 'outputs' / 'R22_v1_0_0'
APO_DIR = PROJECT / 'outputs' / 'R45T_v1_0_0' / 'apo2'
R12B_CKPT = DRIVE_ROOT / 'UMUD_R12B' / 'best_dinov2_frozen.pt'
R12B_SHA = os.environ.get('R71T_R12B_SHA', '21ec34d0f5dd7664ecab0ca608f976fdf9d2a161e2af2873c21d513cd5ae19ef')
SIDE = int(os.environ.get('R71T_SIDE', 768))      # 768 = wejście R22 i rozcięgien v2 (inna wartość tylko w testach)
TRAIN_ROOT = PROJECT / 'data' / 'competition_train_v1'
CFG = dict(r71_train.F71_CONFIG)
CFG.update(json.loads(os.environ.get('R71T_TRAIN_OVERRIDES', '{}')))   # tylko testy lokalne

ver_path = RUN / 'VERIFIED.json'
ver = json.loads(ver_path.read_text()) if ver_path.exists() else {}
def verified(p, sha):
    key = f'{p}|{p.stat().st_size}|{int(p.stat().st_mtime)}'
    if ver.get(key) != sha:
        assert r45_core.sha256(p) == sha, f'checkpoint różni się od zapisanej sumy: {p}'
        ver[key] = sha
    return p

tc = json.loads((R22RUN / 'TRAINING_COMPLETE.json').read_text())
R22_FOLD_PATH = {int(d['fold']): verified(R22RUN / f"fold_{d['fold']}" / 'final.pt', d['checkpoint_sha256']) for d in tc['checkpoints']}
assert sorted(R22_FOLD_PATH) == [0, 1, 2, 3, 4], 'Potrzebne 5 foldów R22'
APO_DONE = [json.loads(p.read_text()) for p in sorted(APO_DIR.glob('fold_*/DONE.json'))]
APO2_PATHS = [verified(APO_DIR / f"fold_{d['fold']}" / 'final.pt', d['final_sha256']) for d in APO_DONE]
assert APO2_PATHS, 'Brak ukończonych foldów rozcięgien v2 (R45T)'
R12B_PATH = verified(R12B_CKPT, R12B_SHA)
r45_core.atomic_json(ver_path, ver)

pairs, prep = r71_train.list_pairs(TRAIN_ROOT / 'fasc_images', TRAIN_ROOT / 'fasc_masks')
FRAME, fold_info = r71_train.folds_from_r22(pairs, R22RUN / 'OOF.csv')
assert len(FRAME) > 0, 'Brak par obraz+maska pęczków zgodnych z OOF.csv R22'
fp = RUN / 'FOLDS.csv'
if fp.exists():
    old = pd.read_csv(fp)
    assert old.name.astype(str).tolist() == FRAME.name.astype(str).tolist() and old.fold.tolist() == FRAME.fold.tolist(), \
        'Inny zbiór / podział niż w poprzedniej sesji — użyj nowego RUN_NAME'
else:
    FRAME[['name', 'file_name', 'fold', 'group_id']].to_csv(fp, index=False)
def load_manifest(csv_path, check_sha):
    m = pd.read_csv(csv_path)
    m['path'] = [str(PROJECT / p) for p in m.relative_path]
    missing = [p for p in m.path if not Path(p).is_file()]
    assert not missing, f'{len(missing)} brakujących obrazów, np. {missing[:3]}'
    assert not m.image_id.duplicated().any(), 'powtórzone image_id w manifeście'
    if check_sha:
        for r in m.itertuples():
            verified(Path(r.path), r.image_sha256)       # assert przy niezgodności; zapamiętane między sesjami
    return m
test = load_manifest(Path(os.environ.get('R71T_TEST_MANIFEST', ASSETS / 'test_manifest.csv')), CHECK_IMAGE_SHA)
osf = load_manifest(Path(os.environ.get('R71T_OSF_MANIFEST', ASSETS / 'osf_manifest.csv')), CHECK_IMAGE_SHA)
r45_core.atomic_json(ver_path, ver)

_cands = [R22RUN / 'pretrained' / 'convnext_tiny_imagenet1k_v1_encoder.pt',
          PROJECT / 'outputs' / 'R45T_v1_0_0' / 'pretrained' / 'convnext_tiny_imagenet1k_v1_encoder.pt']
PRETRAINED = next((p for p in _cands if p.exists()), None)
if PRETRAINED is None:
    PRETRAINED = RUN / 'pretrained' / 'convnext_tiny_imagenet1k_v1_encoder.pt'
    if not PRETRAINED.exists():
        from torchvision.models import convnext_tiny, ConvNeXt_Tiny_Weights
        torch.hub.set_dir(str(RUN / 'pretrained' / 'torch_hub'))
        _m = convnext_tiny(weights=None); _m.load_state_dict(ConvNeXt_Tiny_Weights.IMAGENET1K_V1.get_state_dict(progress=True, check_hash=True))
        r45_models.atomic_torch(PRETRAINED, {'encoder': _m.features.state_dict()})
log('trening pęczków: par', prep['pairs'], '| zgodnych z R22', fold_info['matched'], '| foldy', FRAME.fold.value_counts().sort_index().to_dict(),
    '| test', len(test), '| OSF', len(osf), '| v2 foldów', len(APO2_PATHS))


In [ ]:
# =====================================================================================
# KOMÓRKA 4 — ETAP S: symulator protokołu (R22 poza foldem) + fragmenty testu i OSF (wznawialne)
# =====================================================================================
OUTS = RUN / 'S'; OUTS.mkdir(exist_ok=True)
CACHE = LOCAL / f'r71t_cache_{CFG["side"]}'
CACHE_TAR = RUN / f'cache_{CFG["side"]}.tar'
def write_cache_sample(row, gray):
    dest = CACHE / f'{row.image_id}.npz'
    if dest.exists() or CACHE_TAR.exists():
        return
    CACHE.mkdir(parents=True, exist_ok=True)
    mask = r45_core.read_binary(row.mask)
    square, _ = r45_core.letterbox_image(gray, CFG['side'])
    soft, ang, valid, n_lines = r71_train.make_targets(gray.shape, mask, CFG['side'], CFG['tube_threshold'], CFG['band_px'])
    tmp = dest.with_suffix('.tmp.npz')
    np.savez_compressed(tmp, image=np.rint(square * 255).astype(np.uint8), line=np.rint(soft * 255).astype(np.uint8), ang=ang, valid=valid)
    os.replace(tmp, dest)
    (CACHE / f'{row.image_id}.json').write_text(json.dumps(dict(name=row.image_id, h=int(gray.shape[0]), w=int(gray.shape[1]), n_lines=int(n_lines),
        mask_mismatch=bool(mask.shape != gray.shape), line_frac=float((soft >= CFG['tube_threshold']).mean()))))

if RUN_S and not (OUTS / 'S_SUMMARY.json').exists():
    r22 = {k: r45_models.load_r22(p, DEVICE)[0] for k, p in R22_FOLD_PATH.items()}
    apo2 = [r45_models.load_apo2(p, DEVICE)[0] for p in APO2_PATHS]
    r12b = r45_models.load_r12b(R12B_PATH, DEVICE)
    def restore3(maps, shape):
        return tuple(r45_core.restore_map(z, shape) for z in maps)
    apo_fns = {'r12b': lambda r, gray, sq: r45_models.predict_r12b(r12b, r.path, DEVICE),
               'v2': lambda r, gray, sq: r45_core.restore_map(r45_models.predict_apo2(apo2, sq, DEVICE, True), gray.shape)}
    SIG = {'package': PACKAGE_SHA, 'r22': {k: str(p) for k, p in R22_FOLD_PATH.items()}, 'apo2': [d['final_sha256'] for d in APO_DONE],
           'r12b': R12B_SHA, 'side': SIDE, 'tta': True, 'thresholds': r71_sim.THRESHOLDS}
    rows = FRAME.rename(columns={'name': 'image_id', 'image': 'path'})[['image_id', 'path', 'mask', 'fold', 'group_id']]
    def fasc_oof(r, gray, sq):                        # model foldu, który tego obrazu nie widział
        out = restore3(r45_models.predict_r22([r22[int(r.fold)]], sq, DEVICE, True), gray.shape)
        try:
            write_cache_sample(r, gray)                   # przy okazji cache do etapu T (bez drugiego czytania z Drive)
        except Exception as e:  # noqa: BLE001 - etap T zbuduje brakujące próbki sam
            log('cache: pominięty', r.image_id, type(e).__name__)
        return out
    rec_train = r71_sim.run_images('train', rows, OUTS, fasc_oof, apo_fns, {**SIG, 'set': 'train', 'oof': True}, SIDE, log=log, with_gt=True)
    files = r71_sim.export('train', rec_train, OUTS, 'r22oof')
    fasc_all = lambda r, gray, sq: restore3(r45_models.predict_r22(list(r22.values()), sq, DEVICE, True), gray.shape)
    for name, man in (('test', test), ('osf', osf)):
        recs = r71_sim.run_images(name, man[['image_id', 'path']], OUTS, fasc_all, apo_fns, {**SIG, 'set': name}, SIDE, log=log)
        files.update(r71_sim.export(name, recs, OUTS, 'r22'))
    del r22, apo2, r12b; cuda_gc()
    _im = pd.read_csv(OUTS / 'train_images.csv')
    summary = {'gpu': GPU, 'minutes': round((time.time() - T_START) / 60, 1), 'files_MB': files,
               'train_images': int(len(rec_train)), 'errors': int(sum(r.get('status') != 'OK' for r in rec_train)),
               'mask_size_mismatch': int(((_im.mask_h != _im.h) | (_im.mask_w != _im.w)).sum())}
    if (OUTS / 'train_frags_r22oof_t50.csv.gz').exists():
        f5 = pd.read_csv(OUTS / 'train_frags_r22oof_t50.csv.gz')
        d = np.abs(((f5.angle_fit - f5.angle_or) + 90) % 180 - 90)
        summary.update(frags_per_image_thr05_median=float(f5.groupby('image_id').size().median()) if len(f5) else 0.,
                       angle_fit_vs_or_median_abs_deg=float(np.nanmedian(d)) if len(d) else None)
    if (OUTS / 'train_gt.csv.gz').exists():
        summary['gt_lines_per_image_median'] = float(pd.read_csv(OUTS / 'train_gt.csv.gz').groupby('image_id').size().median())
    r45_core.atomic_json(OUTS / 'S_SUMMARY.json', r45_core.clean_json(summary))
    print(json.dumps(summary, indent=1))
    log('ETAP S GOTOWY ->', OUTS)
elif (OUTS / 'S_SUMMARY.json').exists():
    log('Etap S: już gotowy ->', OUTS)


In [ ]:
# =====================================================================================
# KOMÓRKA 5 — ETAP T: trening F71 (wznawialny) + fragmenty poza foldem + eksporty E
# =====================================================================================
OUTT = RUN / 'T'; OUTT.mkdir(exist_ok=True)

def completed_folds():
    return sorted(int(p.parent.name.split('_')[1]) for p in OUTT.glob('fold_*/DONE.json'))

def oof_export(fold, models, held):
    """Fragments of the fold model on its held-out training images (from the cache, native resolution)."""
    recs = []
    for r in held.itertuples(index=False):
        meta = json.loads((CACHE / f'{r.name}.json').read_text())
        x, _, _, _ = r71_train.load_sample(CACHE, r.name)
        p, c, s = (r45_core.restore_map(z, (meta['h'], meta['w'])) for z in r71_train.predict_maps(models, x, DEVICE, True))
        recs.append({'image_id': r.name, 'h': meta['h'], 'w': meta['w'], 'status': 'OK',
                     'frags': {f'{t:.2f}': r71_geom.frag_lines(p, c, s, t, meta['w']) for t in r71_sim.THRESHOLDS}})
    files = r71_sim.export(f'train_fold{fold}', recs, OUTT / f'fold_{fold}', 'f71oof')
    return {'oof_files_MB': files}

def run_export(k):
    folds = completed_folds()
    if len(folds) < k:
        return None
    out = RUN / f'E{k}'
    if (out / f'E{k}_SUMMARY.json').exists():
        log(f'E{k}: już policzony'); return out
    done = [json.loads((OUTT / f'fold_{f}' / 'DONE.json').read_text()) for f in folds]
    models = [r71_train.load_f71(OUTT / f'fold_{f}' / 'final.pt', DEVICE)[0] for f in folds]
    fasc = lambda r, gray, sq: tuple(r45_core.restore_map(z, gray.shape) for z in r71_train.predict_maps(models, sq, DEVICE, True))
    sig = {'package': PACKAGE_SHA, 'f71': [d['final_sha256'] for d in done], 'side': CFG['side'], 'tta': True}
    files = {}
    for name, man in (('test', test), ('osf', osf)):
        recs = r71_sim.run_images(name, man[['image_id', 'path']], out, fasc, {}, {**sig, 'set': name}, CFG['side'], log=log)
        files.update(r71_sim.export(name, recs, out, f'f71e{k}'))
    del models; cuda_gc()
    r45_core.atomic_json(out / f'E{k}_SUMMARY.json', {'folds': folds, 'gpu': GPU, 'files_MB': files,
                                                     'oof': [d.get('oof_summary') for d in done]})
    log(f'EKSPORT E{k} GOTOWY ->', out)
    return out

if RUN_TRAIN:
    cidx = r71_train.sync_cache(FRAME, CACHE, CACHE_TAR, CFG['side'], CFG, log=log)
    log('cache treningowy:', len(cidx), 'obrazów | maska innego rozmiaru:', int(cidx.mask_mismatch.sum()),
        '| linii na obraz (mediana):', float(cidx.n_lines.median()))
    CFG['_batch'], CFG['_accumulation'] = CFG['batch'], CFG['accumulation']
    deadline = T_START + SESSION_HOURS * 3600
    for fold in FOLDS:
        try:
            d = r71_train.train_fold(FRAME, CACHE, OUTT, PRETRAINED, CFG, fold, DEVICE, log=log, deadline=deadline, after_train=oof_export)
            log(f'fold {fold} gotowy:', json.dumps(d.get('oof_summary')))
        except r71_train.SessionBudgetReached as e:
            log(str(e), '— uruchom wszystko ponownie w nowej sesji'); break
        k = len(completed_folds())
        if k in EXPORT_AFTER:
            run_export(k)
    log('Ukończone foldy F71:', completed_folds(), '| brakujące:', [f for f in FOLDS if f not in completed_folds()])


In [ ]:
# =====================================================================================
# KOMÓRKA 6 — ETAP E (z wszystkimi ukończonymi foldami) i podsumowanie
# =====================================================================================
k = len(completed_folds())
if k and not (RUN / f'E{k}' / f'E{k}_SUMMARY.json').exists():
    run_export(k)
files = sorted(str(p.relative_to(RUN)) for p in RUN.rglob('*') if p.suffix in ('.gz', '.csv', '.json') and 'cache' not in p.name)
print('\n'.join(files))
log(f'GOTOWE. Etap S: {"tak" if (RUN / "S" / "S_SUMMARY.json").exists() else "nie"} | foldy F71: {k}/5. '
    f'Daj znać, który etap się skończył — pliki odczytam z Drive ({RUN}).')
